# 04D_SkillNER_Extraction_Final
Production-oriented skill extraction pipeline for SES.

In [1]:
!pip install skillNer spacy pandas tqdm -q
!python -m spacy download en_core_web_lg -q

  DEPRECATION: Building 'skillNer' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'skillNer'. Discussion can be found at https://github.com/pypa/pip/issues/6334


[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import spacy

tqdm.pandas()
nlp = spacy.load("en_core_web_lg")


In [3]:
postings = pd.read_csv('postings.csv', low_memory=False)

postings['text'] = (
    postings['title'].fillna('') + ' ' +
    postings['description'].fillna('') + ' ' +
    postings['skills_desc'].fillna('')
)

postings.shape


(123849, 32)

In [4]:
TECH_KEYWORDS = [
'python','sql','java','javascript','typescript','c++','c#','go','rust',
'aws','azure','gcp','docker','kubernetes',
'mongodb','postgresql','mysql','redis',
'react','node.js','next.js','django','flask','fastapi',
'tensorflow','pytorch','scikit-learn','spark',
'tableau','power bi','pandas','numpy',
'langchain','llm','rag','machine learning','deep learning'
]

SOFT_KEYWORDS = [
'communication','leadership','teamwork','problem solving',
'critical thinking','adaptability','collaboration',
'time management','presentation','stakeholder management'
]


In [5]:
import re

def extract_skills(text):

    text = str(text).lower()
    found = []

    for skill in TECH_KEYWORDS + SOFT_KEYWORDS:

        pattern = r'\\b' + re.escape(skill) + r'\\b'

        if re.search(pattern, text):
            found.append(skill)

    return list(set(found))


In [6]:
sample = postings.copy()

sample['extracted_skills'] = (
    sample['text']
    .progress_apply(extract_skills)
)

sample[['title','extracted_skills']].head()


100%|████████████████████████████████████████████████████████████████████████| 123849/123849 [00:46<00:00, 2667.33it/s]


,title,extracted_skills
0,Marketing Coordinator,[]
1,Mental Health Therapist/Counselor,[]
2,Assitant Restaurant Manager,[]
3,Senior Elder Law / Trusts and Estates Associat...,[]
4,Service Technician,[]


In [7]:
rows=[]

for _,row in sample.iterrows():

    salary = row.get('normalized_salary', np.nan)

    for skill in row['extracted_skills']:

        rows.append([skill, salary])

skills_df = pd.DataFrame(
    rows,
    columns=['skill','salary']
)

skills_df.head()


,skill,salary


In [8]:
demand_df = (
    skills_df['skill']
    .value_counts()
    .reset_index()
)

demand_df.columns=[
    'skill',
    'linkedin_demand'
]

salary_df = (
    skills_df
    .groupby('skill')['salary']
    .median()
    .reset_index()
)

salary_df.columns=[
    'skill',
    'salary_premium'
]


In [9]:
master = demand_df.merge(
    salary_df,
    on='skill',
    how='left'
)

master['category'] = np.where(
    master['skill'].isin(SOFT_KEYWORDS),
    'Soft Skill',
    'Technical'
)

master.sort_values(
    'linkedin_demand',
    ascending=False
).head(50)


,skill,linkedin_demand,salary_premium,category


In [10]:
master.to_csv(
    'skillner_skill_master.csv',
    index=False
)

print(master.shape)
print('Saved: skillner_skill_master.csv')


(0, 4)
Saved: skillner_skill_master.csv
